# 12 — Persistência de modelos e reprodutibilidade

**EP · Data Science Knowledge Base | Deep Learning | Notebook 12/20**


**Pré-requisitos:** Python, NumPy, fundamentos de aprendizado supervisionado; JupyterLab; TensorFlow/Keras apenas nos laboratórios indicados.

**Escopo de validação:** os exemplos básicos podem ser reproduzidos sem TensorFlow; laboratórios opcionais precisam de validação local independente.

---

**Deep Learning · Capítulo de estudo independente**

> Material didático autoral · estudo independente · exemplos locais e laboratórios opcionais

**Referência curricular:** `material histórico de referência`. O material original foi preservado; este capítulo é uma reconstrução didática independente.

### Materiais complementares

- [Guia Técnico](./reference_materials/12_persistencia_reproducibilidade.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_persistencia_reproducibilidade.pdf)


## Como estudar este capítulo

1. Leia a teoria e deduza as expressões apresentadas.
2. Execute as células do **núcleo obrigatório** em um kernel limpo.
3. Confira a interpretação e responda ao exercício antes de abrir a solução.
4. Se tiver TensorFlow e Keras compatíveis, execute o **laboratório opcional**.

**Pré-requisitos transversais:** Python (01), Estatística (03), preparação/validação de ML (05); aplicações textuais remetem à PLN (06). Não há necessidade de repetir integralmente esses cursos.

## Objetivos de aprendizagem

- Diferenciar serialização de modelo, pesos e pipeline completo.
- Relacionar a técnica à sua função de perda, representação e protocolo de avaliação.
- Explicar limitações e formular um teste reprodutível.

## 1. Fundamentação conceitual

O formato Keras moderno `.keras` salva configuração, pesos e estado do otimizador quando aplicável. Arquivos `.h5` históricos requerem compatibilidade. Preservar versão das dependências, transformações, sementes, métricas e esquema de dados. Nunca carregar modelos de fontes não confiáveis.

**Fonte histórica de apoio:** notas do curso original, consultadas apenas como referência de contexto; as explicações e exemplos desta edição são independentes.

**Questões críticas:** quais pressupostos são necessários? O que a função aprende? Qual métrica indica utilidade real? Que tipo de vazamento ou má interpretação invalidaria a conclusão?

## 2. Matemática essencial

**Expressão esquemática:** `artefato = arquitetura + pesos + estado`

A expressão acima resume a relação central. Para a derivação, identifique dimensões, hipóteses e operações; não interprete notação compacta como uma implementação completa.

## 3. Experimento obrigatório — NumPy e/ou scikit-learn

Este experimento é propositalmente pequeno: **executa sem GPU, internet ou serviço externo**. Quando há treino de baseline, não interpretá-lo como evidência da qualidade de uma rede neural.

In [1]:
import json, numpy as np
metadata={'seed':42,'split':'estratificado','metrica':'f1','framework':'registro didático'}
encoded=json.dumps(metadata,ensure_ascii=False)
assert json.loads(encoded)['seed']==42
print('Manifesto de experimento:',encoded)

Manifesto de experimento: {"seed": 42, "split": "estratificado", "metrica": "f1", "framework": "registro didático"}


### Interpretação e limitações

Compare o resultado obtido com a formulação matemática. Confirme shapes, escala e hipótese de modelagem. **Um experimento sintético valida uma operação, não substitui avaliação em dados reais.** Ao treinar modelos, separe dados e ajuste transformações apenas no treino; preserve um conjunto final de teste.

## 4. Laboratório neural opcional — TensorFlow / Keras 3

Esta seção demonstra **APIs reais**, mas depende de versões compatíveis e pode exigir mais recursos que o núcleo. Não é necessária para executar o notebook básico. Não se deve confundir um teste de chamada de API com avaliação experimental de desempenho. Se o pacote TensorFlow não estiver disponível, será exibido um aviso de não execução.

In [2]:
# Laboratório opcional: execute após instalar e configurar TensorFlow/Keras compatíveis.
# Se TensorFlow estiver ausente, o exemplo é ignorado com aviso explícito.
import importlib.util
if importlib.util.find_spec('tensorflow') is None:
    print('Laboratório TensorFlow/Keras não executado: TensorFlow ausente neste kernel.')
else:
    import keras, tempfile
    from keras import layers
    modelo=keras.Sequential([keras.Input(shape=(3,)),layers.Dense(1)]);modelo.compile(optimizer='adam',loss='mse')
    with tempfile.TemporaryDirectory() as tmp:
     caminho=tmp+'/modelo.keras';modelo.save(caminho);recuperado=keras.models.load_model(caminho)
     print('Modelo recarregado:',recuperado.count_params())

Laboratório TensorFlow/Keras não executado: TensorFlow ausente neste kernel.


## 5. Exercício de fixação

**Questão:** Salvar apenas pesos basta para reproduzir o pipeline?

<details>
<summary>Confira a solução comentada</summary>

**Solução:** Não; também são necessários arquitetura, pré-processamento, hiperparâmetros e versões.

**Verificação:** explique o resultado usando notação, shapes e consequências para generalização.
</details>

## 6. Erros comuns e boas práticas

- Confundir ajuste no treino com generalização para dados não vistos.
- Reportar somente uma métrica ou uma execução aleatória.
- Ignorar shape, tipo de variável e convenção do backend.
- Executar exemplos com pesos baixados sem registrar versão, licença e procedência.
- Confundir demonstração matemática com validação de um modelo de produção.

## 7. Resumo de consulta rápida

**Ideia central:** O formato Keras moderno `.

**Expressão:** `artefato = arquitetura + pesos + estado`.

**Lembrete:** começar por baseline; respeitar separação de dados; documentar decisões; avaliar limitações.

**Ponte com outros cursos:** ML I (05) para métricas e validação; PLN (06) para modelos de linguagem; projetos multidisciplinares ficam reservados à fase posterior.

**Fontes de atualização (consultar quando necessário):** [Keras 3](https://keras.io/guides/), [TensorFlow](https://www.tensorflow.org/guide), [scikit-learn](https://scikit-learn.org/stable/user_guide.html).

## Laboratório aplicado revisado — Persistência, restauração e identidade das previsões

Persistência inclui modelo, pré-processamento, versões e comportamento após recarga; carregamentos inseguros podem executar código.

**Procedimento:** execute a próxima célula com o núcleo declarado em `requirements.txt`, confira os `assert` e interprete os resultados. O experimento é independente dos exemplos anteriores.

In [3]:
import tempfile
from pathlib import Path
import joblib,numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X=np.array([[0,0],[0,1],[1,0],[1,1],[2,0],[0,2]],dtype=float)
y=np.array([0,0,0,1,1,1]);m=make_pipeline(StandardScaler(),LogisticRegression()).fit(X,y)
with tempfile.TemporaryDirectory() as d:
    fp=Path(d)/'modelo.joblib';joblib.dump(m,fp);rec=joblib.load(fp)
    assert np.allclose(m.predict_proba(X),rec.predict_proba(X))
    print('Modelo restaurado, probabilidades idênticas, tamanho:',fp.stat().st_size,'bytes')

Modelo restaurado, probabilidades idênticas, tamanho: 1409 bytes


### Exportação de resultados (local)

Execute esta célula após o experimento acima. Os arquivos são salvos em `outputs/` e não entram automaticamente no Git.

In [4]:
# Exemplo persistente: carregar arquivos joblib SOMENTE de origens confiáveis.
# A célula anterior treinou 'm'; os dados originais permanecem inalterados.
from pathlib import Path
import json
import joblib
import numpy as np

out=Path('outputs'); out.mkdir(exist_ok=True)
modelo_path=out/'12_pipeline_logistic_regression.joblib'
manifesto_path=out/'12_pipeline_metadata.json'
joblib.dump(m,modelo_path)
manifesto={'capitulo':12,'tipo':'sklearn_pipeline','formato':'joblib',
           'aviso':'Arquivos joblib não confiáveis podem executar código ao carregar.'}
manifesto_path.write_text(json.dumps(manifesto,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
restaurado=joblib.load(modelo_path)  # apenas arquivo salvo pela própria célula
assert np.allclose(m.predict_proba(X),restaurado.predict_proba(X))
print('Modelo e manifesto:',modelo_path,manifesto_path)


Modelo e manifesto: outputs/12_pipeline_logistic_regression.joblib outputs/12_pipeline_metadata.json


### Interpretação, limites e verificação

Nunca carregue modelos serializados de fonte não confiável. Registre versão das bibliotecas, parâmetros de treino e métricas além do artefato.

**Checklist:** conferir entradas e saídas, sementes, inexistência de uso dos dados de teste durante ajuste e distinção entre uma demonstração sintética e desempenho em aplicações reais.